# ELT - NYC Yellow Taxi (Janeiro/2024)

Esqueleto do pipeline. Estrutura sugerida:

1. Setup e conexão
2. Carga da camada raw (dados brutos, sem transformação)
3. Verificação básica da carga
4. Perfilamento dos dados 
5. Camada de tratamento / staging 
6. Modelo dimensional 
7. Camada de consumo analítico 

## 1. Setup e conexão

In [1]:
import duckdb
import os

# Caminhos dos arquivos de origem — ajustem se estiverem em outra pasta
PARQUET_PATH = "../data/yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_PATH = "../data/taxi_zone_lookup.csv"

# Arquivo de banco persistente em disco (fica salvo entre execuções)
DB_PATH = "nyc_taxi.duckdb"

con = duckdb.connect(DB_PATH)

## 2. Carga da camada raw (dados brutos, sem transformação)

A camada raw deve ser uma cópia fiel do
que chegou, para fins de auditoria e rastreabilidade. 

In [2]:
con.execute(f"""
    CREATE OR REPLACE TABLE raw_yellow_tripdata AS
    SELECT *
    FROM read_parquet('{PARQUET_PATH}')
""")

In [3]:
con.execute(f"""
    CREATE OR REPLACE TABLE raw_taxi_zone_lookup AS
    SELECT *
    FROM read_csv_auto('{ZONE_LOOKUP_PATH}')
""")

## 3. Verificação básica da carga


In [4]:
con.execute("SELECT COUNT(*) AS total_linhas FROM raw_yellow_tripdata").fetchdf()

,total_linhas
0,2964624


In [5]:
con.execute("SELECT * FROM raw_yellow_tripdata LIMIT 5").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [6]:
con.execute("SELECT COUNT(*) AS total_linhas FROM raw_taxi_zone_lookup").fetchdf()

,total_linhas
0,265


In [7]:
con.execute("SELECT * FROM raw_taxi_zone_lookup LIMIT 5").fetchdf()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [8]:
con.execute("DESCRIBE raw_yellow_tripdata").fetchdf()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [9]:
con.execute("DESCRIBE raw_taxi_zone_lookup").fetchdf()

,column_name,column_type,null,key,default,extra
0,LocationID,BIGINT,YES,None,None,None
1,Borough,VARCHAR,YES,None,None,None
2,Zone,VARCHAR,YES,None,None,None
3,service_zone,VARCHAR,YES,None,None,None


## 4. Perfilamento dos dados

Coisas a investigar em `raw_yellow_tripdata` e `raw_taxi_zone_lookup`:
- Nulos por coluna
- Duplicidade de registros
- Cardinalidade e domínio das colunas categóricas (ex.: payment_type, RatecodeID)
- Valores improváveis (datas fora de janeiro/2024, distâncias/valores negativos, etc.)
- Correspondência dos códigos de localização entre os dois arquivos

Guardem aqui as consultas usadas e o que cada uma revelou.

# TODO: Perfilamento

### 4.1 Valores Nulos

In [10]:
con.execute("""
    SELECT COUNT(*) AS total_linhas,
           COUNT(*) - COUNT(VendorID) AS nulos_vendorid,
           COUNT(*) - COUNT(tpep_pickup_datetime) AS nulos_pickup_datetime,
           COUNT(*) - COUNT(tpep_dropoff_datetime) AS nulos_dropoff_datetime,
           COUNT(*) - COUNT(passenger_count) AS nulos_passenger_count,
           COUNT(*) - COUNT(trip_distance) AS nulos_trip_distance,
           COUNT(*) - COUNT(RatecodeID) AS nulos_ratecodeid,
           COUNT(*) - COUNT(store_and_fwd_flag) AS nulos_store_and_fwd_flag,
           COUNT(*) - COUNT(PULocationID) AS nulos_pulocationid,
           COUNT(*) - COUNT(DOLocationID) AS nulos_dolocationid,
           COUNT(*) - COUNT(payment_type) AS nulos_payment_type,
           COUNT(*) - COUNT(fare_amount) AS nulos_fare_amount,
           COUNT(*) - COUNT(extra) AS nulos_extra,
           COUNT(*) - COUNT(mta_tax) AS nulos_mta_tax,
           COUNT(*) - COUNT(tip_amount) AS nulos_tip_amount,
           COUNT(*) - COUNT(tolls_amount) AS nulos_tolls_amount,
           COUNT(*) - COUNT(improvement_surcharge) AS nulos_improvement_surcharge,
           COUNT(*) - COUNT(total_amount) AS nulos_total_amount,
           COUNT(*) - COUNT(congestion_surcharge) AS nulos_congestion_surcharge,
           COUNT(*) - COUNT(Airport_fee) AS nulos_airport_fee
    FROM raw_yellow_tripdata
""").fetchdf()

,total_linhas,nulos_vendorid,nulos_pickup_datetime,nulos_dropoff_datetime,nulos_passenger_count,nulos_trip_distance,nulos_ratecodeid,nulos_store_and_fwd_flag,nulos_pulocationid,nulos_dolocationid,nulos_payment_type,nulos_fare_amount,nulos_extra,nulos_mta_tax,nulos_tip_amount,nulos_tolls_amount,nulos_improvement_surcharge,nulos_total_amount,nulos_congestion_surcharge,nulos_airport_fee
0,2964624,0,0,0,140162,0,140162,140162,0,0,0,0,0,0,0,0,0,0,140162,140162


In [11]:
con.execute("""
    SELECT
        COUNT(*) AS linhas_com_todas_nulas
    FROM raw_yellow_tripdata
    WHERE passenger_count IS NULL
      AND RatecodeID IS NULL
      AND store_and_fwd_flag IS NULL
      AND congestion_surcharge IS NULL
      AND Airport_fee IS NULL
""").fetchdf()

,linhas_com_todas_nulas
0,140162


### 4.2 Linhas Duplicadas

In [12]:
con.execute("""
    SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
           PULocationID, DOLocationID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID
    HAVING COUNT(*) > 1
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,qtd
0,2,2024-01-20 23:30:55,2024-01-20 23:42:09,50,230,2
1,2,2024-01-21 00:56:54,2024-01-21 01:28:14,79,166,2
2,2,2024-01-21 02:06:10,2024-01-21 02:18:17,48,41,2
3,2,2024-01-21 02:42:07,2024-01-21 02:48:38,68,186,2
4,2,2024-01-21 02:29:40,2024-01-21 02:31:25,79,79,2
...,...,...,...,...,...,...
31104,2,2024-01-30 17:43:37,2024-01-30 17:45:45,138,223,2
31105,2,2024-01-30 17:17:52,2024-01-30 17:27:31,237,161,2
31106,2,2024-01-30 18:07:04,2024-01-30 18:17:47,100,246,2
31107,2,2024-01-30 19:27:06,2024-01-30 19:30:23,236,262,2


### 4.3 Payment_type

In [13]:
con.execute("""
    SELECT payment_type, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY payment_type
    ORDER BY qtd DESC
""").fetchdf()

,payment_type,qtd
0,1,2319046
1,2,439191
2,0,140162
3,4,46628
4,3,19597


### 4.4 RatecodeID

In [14]:
con.execute("""
    SELECT RatecodeID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY RatecodeID
    ORDER BY qtd DESC
""").fetchdf()

,RatecodeID,qtd
0,1,2663350
1,<NA>,140162
2,2,98713
3,99,28663
4,5,19410
5,3,7954
6,4,6365
7,6,7


### 4.5 VendorID

In [15]:
con.execute("""
    SELECT VendorID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY VendorID
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,qtd
0,2,2234632
1,1,729732
2,6,260


### 4.6 Passenger_count

In [16]:
con.execute("""
    SELECT passenger_count, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY passenger_count
    ORDER BY passenger_count
""").fetchdf()

,passenger_count,qtd
0,0,31465
1,1,2188739
2,2,405103
3,3,91262
4,4,51974
5,5,33506
6,6,22353
7,7,8
8,8,51
9,9,1


### 4.7 store_and_fwd_flag

In [17]:
con.execute("""
    SELECT store_and_fwd_flag, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    GROUP BY store_and_fwd_flag
    ORDER BY qtd DESC
""").fetchdf()

,store_and_fwd_flag,qtd
0,N,2813126
1,NaN,140162
2,Y,11336


---

### 4.8 Valores max/min

In [18]:
con.execute("""
    SELECT
        MIN(trip_distance) AS min_distancia, MAX(trip_distance) AS max_distancia,
        MIN(fare_amount) AS min_tarifa, MAX(fare_amount) AS max_tarifa,
        MIN(extra) AS min_extra, MAX(extra) AS max_extra,
        MIN(mta_tax) AS min_mta_tax, MAX(mta_tax) AS max_mta_tax,
        MIN(tip_amount) AS min_tip, MAX(tip_amount) AS max_tip,
        MIN(tolls_amount) AS min_tolls, MAX(tolls_amount) AS max_tolls,
        MIN(improvement_surcharge) AS min_improvement, MAX(improvement_surcharge) AS max_improvement,
        MIN(total_amount) AS min_total, MAX(total_amount) AS max_total,
        MIN(congestion_surcharge) AS min_congestion, MAX(congestion_surcharge) AS max_congestion,
        MIN(Airport_fee) AS min_airport_fee, MAX(Airport_fee) AS max_airport_fee
    FROM raw_yellow_tripdata
""").fetchdf().T

,0
min_distancia,0.00
max_distancia,312722.30
min_tarifa,-899.00
max_tarifa,5000.00
min_extra,-7.50
max_extra,14.25
min_mta_tax,-0.50
max_mta_tax,4.00
min_tip,-80.00
max_tip,428.00


### 4.8 Média e Mediana

In [19]:
con.execute("""
    SELECT
        ROUND(AVG(trip_distance), 2) AS media_distancia, MEDIAN(trip_distance) AS mediana_distancia,
        ROUND(AVG(fare_amount), 2) AS media_tarifa, MEDIAN(fare_amount) AS mediana_tarifa,
        ROUND(AVG(extra), 2) AS media_extra, MEDIAN(extra) AS mediana_extra,
        ROUND(AVG(mta_tax), 2) AS media_mta_tax, MEDIAN(mta_tax) AS mediana_mta_tax,
        ROUND(AVG(tip_amount), 2) AS media_tip, MEDIAN(tip_amount) AS mediana_tip,
        ROUND(AVG(tolls_amount), 2) AS media_tolls, MEDIAN(tolls_amount) AS mediana_tolls,
        ROUND(AVG(improvement_surcharge), 2) AS media_improvement, MEDIAN(improvement_surcharge) AS mediana_improvement,
        ROUND(AVG(total_amount), 2) AS media_total, MEDIAN(total_amount) AS mediana_total,
        ROUND(AVG(congestion_surcharge), 2) AS media_congestion, MEDIAN(congestion_surcharge) AS mediana_congestion,
        ROUND(AVG(Airport_fee), 2) AS media_airport_fee, MEDIAN(Airport_fee) AS mediana_airport_fee,
        ROUND(AVG(passenger_count), 2) AS media_passageiros
    FROM raw_yellow_tripdata
""").fetchdf().T

,0
media_distancia,3.65
mediana_distancia,1.68
media_tarifa,18.18
mediana_tarifa,12.80
media_extra,1.45
mediana_extra,1.00
media_mta_tax,0.48
mediana_mta_tax,0.50
media_tip,3.34
mediana_tip,2.70


---

### 4.9 Desembarque antes do Embarque

In [20]:
con.execute("""
    SELECT COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    WHERE tpep_dropoff_datetime < tpep_pickup_datetime
""").fetchdf()

,qtd
0,56


In [21]:
con.execute("""
    SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
           trip_distance, fare_amount, PULocationID, DOLocationID
    FROM raw_yellow_tripdata
    WHERE tpep_dropoff_datetime < tpep_pickup_datetime
    ORDER BY tpep_pickup_datetime
    LIMIT 10
""").fetchdf()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,PULocationID,DOLocationID
0,6,2024-01-01 02:01:50,2024-01-01 02:01:39,7.46,39.22,265,197
1,6,2024-01-01 03:01:46,2024-01-01 03:01:39,7.65,40.69,265,62
2,1,2024-01-02 10:00:00,2024-01-02 09:53:56,7.20,38.50,75,95
3,6,2024-01-02 16:01:30,2024-01-02 16:01:19,9.02,49.82,265,95
4,6,2024-01-02 16:01:44,2024-01-02 16:01:24,7.32,44.20,265,129
5,6,2024-01-03 05:01:56,2024-01-03 05:01:27,4.63,18.43,265,130
6,6,2024-01-03 16:01:34,2024-01-03 16:01:21,3.90,31.20,265,260
7,6,2024-01-03 18:01:49,2024-01-03 18:01:44,8.09,39.20,265,168
8,1,2024-01-04 12:30:00,2024-01-04 12:18:23,4.80,31.50,165,222
9,1,2024-01-04 14:00:00,2024-01-04 13:55:50,5.60,27.50,95,80


In [22]:
con.execute("""
    SELECT VendorID, COUNT(*) AS qtd
    FROM raw_yellow_tripdata
    WHERE tpep_dropoff_datetime < tpep_pickup_datetime
    GROUP BY VendorID
    ORDER BY qtd DESC
""").fetchdf()

,VendorID,qtd
0,6,48
1,1,8


---

### 4.10 Outliers

In [23]:
con.execute("""
    SELECT
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(trip_distance, 0.25) AS q1,
                        quantile_cont(trip_distance, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                trip_distance < l.q1 - 1.5 * (l.q3 - l.q1)
                OR trip_distance > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_trip_distance,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(fare_amount, 0.25) AS q1,
                        quantile_cont(fare_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                fare_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR fare_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_fare_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(extra, 0.25) AS q1,
                        quantile_cont(extra, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                extra < l.q1 - 1.5 * (l.q3 - l.q1)
                OR extra > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_extra,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(mta_tax, 0.25) AS q1,
                        quantile_cont(mta_tax, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                mta_tax < l.q1 - 1.5 * (l.q3 - l.q1)
                OR mta_tax > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_mta_tax,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(tip_amount, 0.25) AS q1,
                        quantile_cont(tip_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                tip_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR tip_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_tip_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(tolls_amount, 0.25) AS q1,
                        quantile_cont(tolls_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                tolls_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR tolls_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_tolls_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(improvement_surcharge, 0.25) AS q1,
                        quantile_cont(improvement_surcharge, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                improvement_surcharge < l.q1 - 1.5 * (l.q3 - l.q1)
                OR improvement_surcharge > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_improvement_surcharge,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(total_amount, 0.25) AS q1,
                        quantile_cont(total_amount, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                total_amount < l.q1 - 1.5 * (l.q3 - l.q1)
                OR total_amount > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_total_amount,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(congestion_surcharge, 0.25) AS q1,
                        quantile_cont(congestion_surcharge, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                congestion_surcharge < l.q1 - 1.5 * (l.q3 - l.q1)
                OR congestion_surcharge > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_congestion_surcharge,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(Airport_fee, 0.25) AS q1,
                        quantile_cont(Airport_fee, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                Airport_fee < l.q1 - 1.5 * (l.q3 - l.q1)
                OR Airport_fee > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_airport_fee,
        
        (
            SELECT
                COUNT(*)
            FROM
                raw_yellow_tripdata,
                (
                    SELECT
                        quantile_cont(passenger_count, 0.25) AS q1,
                        quantile_cont(passenger_count, 0.75) AS q3
                    FROM
                        raw_yellow_tripdata
                ) l
            WHERE
                passenger_count < l.q1 - 1.5 * (l.q3 - l.q1)
                OR passenger_count > l.q3 + 1.5 * (l.q3 - l.q1)
        ) AS outliers_passenger_count
""").fetchdf().T

,0
outliers_trip_distance,382745
outliers_fare_amount,318801
outliers_extra,47566
outliers_mta_tax,64150
outliers_tip_amount,216756
outliers_tolls_amount,210815
outliers_improvement_surcharge,36914
outliers_total_amount,363621
outliers_congestion_surcharge,246707
outliers_airport_fee,237673


## 5. Camada de tratamento (staging)

### -Staging: Zonas

In [24]:
con.execute("""
    CREATE TABLE stg_zones AS
    SELECT DISTINCT *
    FROM raw_taxi_zone_lookup
""")

### -Staging: Taxi

In [25]:
con.execute("""
    CREATE TABLE stg_taxi AS
    SELECT DISTINCT *
    FROM raw_yellow_tripdata
    WHERE
        trip_distance > 0
        AND tpep_dropoff_datetime > tpep_pickup_datetime
        AND passenger_count IS NOT NULL
""")

## 6. Modelo dimensional

### Limpeza Prévia

In [26]:
con.execute("DROP TABLE IF EXISTS dim_localizacao")
con.execute("DROP TABLE IF EXISTS dim_pagamento")
con.execute("DROP TABLE IF EXISTS dim_tarifa")
con.execute("DROP TABLE IF EXISTS dim_data")
con.execute("DROP TABLE IF EXISTS fato_corridas")

---

### Localização

In [27]:
con.execute("""
    CREATE TABLE dim_localizacao AS
    SELECT LocationID, Borough, Zone, service_zone
    FROM stg_zones
""")

---

### Pagamento

In [28]:
con.execute("""
    CREATE TABLE dim_pagamento AS
    SELECT DISTINCT
        payment_type,
        CASE payment_type
            WHEN 1 THEN 'Credit Card'
            WHEN 2 THEN 'Cash'
            WHEN 3 THEN 'No Charge'
            WHEN 4 THEN 'Dispute'
            WHEN 5 THEN 'Unknown'
            WHEN 6 THEN 'Voided Trip'
            ELSE 'Não informado'
        END AS descricao
    FROM stg_taxi
""")

---

### Tarifa

In [29]:
con.execute("""
    CREATE TABLE dim_tarifa AS
    SELECT DISTINCT
        RatecodeID,
        CASE RatecodeID
            WHEN 1 THEN 'Standard Rate'
            WHEN 2 THEN 'JFK'
            WHEN 3 THEN 'Newark'
            WHEN 4 THEN 'Nassau/Westchester'
            WHEN 5 THEN 'Negotiated Fare'
            WHEN 6 THEN 'Group Ride'
            WHEN 99 THEN 'Unknown'
            ELSE 'Não informado'
        END AS descricao
    FROM stg_taxi
""")

---

### Data

In [30]:
con.execute("""
    CREATE TABLE dim_data AS
    SELECT DISTINCT
        CAST(tpep_pickup_datetime AS DATE) AS data,
        YEAR(tpep_pickup_datetime) AS ano,
        MONTH(tpep_pickup_datetime) AS mes,
        DAY(tpep_pickup_datetime) AS dia,
        HOUR(tpep_pickup_datetime) AS hora,
        DAYNAME(tpep_pickup_datetime) AS dia_semana
    FROM stg_taxi
""")

---


### Tabela Fato - Corridas

In [31]:
con.execute("""
    CREATE TABLE fato_corridas AS
    SELECT
        VendorID,
        CAST(tpep_pickup_datetime AS DATE) AS data,
        payment_type,
        RatecodeID,
        PULocationID,
        DOLocationID,
        passenger_count,
        trip_distance,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        congestion_surcharge,
        Airport_fee,
        total_amount
    FROM stg_taxi
""")

## 7. Camada de consumo analítico


### 7.1 - Zonas com mais embarque/desembarque

In [32]:
con.execute("""
    SELECT
        z.Zone,
        z.Borough,
        COUNT(*) AS qtd_embarques
    FROM fato_corridas f
    JOIN dim_localizacao z ON f.PULocationID = z.LocationID
    GROUP BY z.Zone, z.Borough
    ORDER BY qtd_embarques DESC
    LIMIT 10
""").fetchdf()

,Zone,Borough,qtd_embarques
0,JFK Airport,Queens,141746
1,Upper East Side South,Manhattan,138396
2,Midtown Center,Manhattan,138050
3,Upper East Side North,Manhattan,130602
4,Midtown East,Manhattan,103431
5,Penn Station/Madison Sq West,Manhattan,101873
6,Times Sq/Theatre District,Manhattan,101534
7,Lincoln Square East,Manhattan,99894
8,LaGuardia Airport,Queens,88084
9,Upper West Side South,Manhattan,83101


In [33]:
con.execute("""
    SELECT
        z.Zone,
        z.Borough,
        COUNT(*) AS qtd_desembarques
    FROM fato_corridas f
    JOIN dim_localizacao z ON f.DOLocationID = z.LocationID
    GROUP BY z.Zone, z.Borough
    ORDER BY qtd_desembarques DESC
    LIMIT 10
""").fetchdf()

,Zone,Borough,qtd_desembarques
0,Upper East Side North,Manhattan,136281
1,Upper East Side South,Manhattan,125448
2,Midtown Center,Manhattan,107200
3,Times Sq/Theatre District,Manhattan,86332
4,Lincoln Square East,Manhattan,85563
5,Upper West Side South,Manhattan,84656
6,Murray Hill,Manhattan,82498
7,Midtown East,Manhattan,81467
8,Lenox Hill West,Manhattan,79754
9,East Chelsea,Manhattan,70207


---

### 7.2 - Corridas com 1 passageiro

In [34]:
con.execute("""
    SELECT
        COUNT(*) AS total_corridas,
        SUM(CASE WHEN passenger_count = 1 THEN 1 ELSE 0 END) AS corridas_1_passageiro,
        ROUND(SUM(CASE WHEN passenger_count = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS porcentagem
    FROM fato_corridas
""").fetchdf()

,total_corridas,corridas_1_passageiro,porcentagem
0,2786858,2159688.0,77.5


---

### 7.3 - Top 10 destinos com corridas mais caras/demoradas

In [35]:
con.execute("""
    SELECT
        z.Zone AS zona_destino,
        z.Borough,
        COUNT(*) AS qtd_corridas,
        ROUND(AVG(f.total_amount), 2) AS valor_medio
    FROM fato_corridas f
    JOIN dim_localizacao z ON f.DOLocationID = z.LocationID
    GROUP BY z.Zone, z.Borough
    HAVING COUNT(*) >= 30
    ORDER BY valor_medio DESC
    LIMIT 10
""").fetchdf()

,zona_destino,Borough,qtd_corridas,valor_medio
0,Newark Airport,EWR,6495,125.43
1,Westerleigh,Staten Island,32,122.39
2,Bloomfield/Emerson Hill,Staten Island,52,116.82
3,Outside of NYC,N/A,10907,116.46
4,Mariners Harbor,Staten Island,36,113.39
5,Saint George/New Brighton,Staten Island,59,111.41
6,Heartland Village/Todt Hill,Staten Island,62,109.77
7,New Dorp/Midland Beach,Staten Island,30,109.18
8,Grymes Hill/Clifton,Staten Island,34,108.62
9,West Brighton,Staten Island,40,95.57


In [36]:
con.execute("""
    SELECT
        z.Zone AS zona_destino,
        z.Borough,
        COUNT(*) AS qtd_corridas,
        ROUND(AVG(EPOCH(t.tpep_dropoff_datetime - t.tpep_pickup_datetime)) / 60, 2) AS duracao_media_minutos
    FROM stg_taxi t
    JOIN dim_localizacao z ON t.DOLocationID = z.LocationID
    GROUP BY z.Zone, z.Borough
    HAVING COUNT(*) >= 30
    ORDER BY duracao_media_minutos DESC
    LIMIT 10
""").fetchdf()

,zona_destino,Borough,qtd_corridas,duracao_media_minutos
0,New Dorp/Midland Beach,Staten Island,30,77.40
1,Arrochar/Fort Wadsworth,Staten Island,58,57.58
2,Hammels/Arverne,Queens,350,49.50
3,Saint George/New Brighton,Staten Island,59,48.06
4,Grymes Hill/Clifton,Staten Island,34,46.33
5,South Beach/Dongan Hills,Staten Island,44,44.54
6,Erasmus,Brooklyn,431,44.22
7,Borough Park,Brooklyn,866,43.82
8,Williamsbridge/Olinville,Bronx,464,43.34
9,Coney Island,Brooklyn,485,42.93


----
### 7.4 Top 10 zonas mais caras

In [37]:
con.execute("""
SELECT
    l.Zone AS zona,
    l.Borough AS distrito,
    ROUND(AVG(f.total_amount), 2) AS valor_medio_corrida,
    COUNT(*) AS total_corridas
FROM fato_corridas f
JOIN dim_localizacao l
    ON f.PULocationID = l.LocationID
GROUP BY
    l.Zone,
    l.Borough
HAVING COUNT(*) >= 100
ORDER BY valor_medio_corrida DESC
LIMIT 10;
""").fetchdf()

,zona,distrito,valor_medio_corrida,total_corridas
0,Outside of NYC,N/A,84.46,506
1,JFK Airport,Queens,77.30,141746
2,Baisley Park,Queens,73.02,837
3,Springfield Gardens South,Queens,71.85,387
4,Flushing Meadows-Corona Park,Queens,69.18,481
5,LaGuardia Airport,Queens,65.26,88084
6,South Ozone Park,Queens,64.34,549
7,East Elmhurst,Queens,62.97,11968
8,South Jamaica,Queens,62.08,512
9,Spuyten Duyvil/Kingsbridge,Bronx,59.26,106


----
### 7.5 formas de pagamentos mais utilizadas

In [38]:
con.execute("""
SELECT
    p.descricao AS forma_pagamento,
    COUNT(*) AS total_corridas
FROM fato_corridas f
JOIN dim_pagamento p
ON f.payment_type = p.payment_type
GROUP BY p.descricao
ORDER BY total_corridas DESC;
""").fetchdf()

,forma_pagamento,total_corridas
0,Credit Card,2298401
1,Cash,430592
2,Dispute,42835
3,No Charge,15030


----
### 7.6 Dias com mais corridas

In [39]:
con.execute("""
SELECT
    d.dia_semana,
    COUNT(*) AS total_corridas
FROM fato_corridas f
JOIN dim_data d
ON f.data = d.data
GROUP BY d.dia_semana
ORDER BY total_corridas DESC;
""").fetchdf()

,dia_semana,total_corridas
0,Wednesday,11252136
1,Tuesday,10449578
2,Thursday,9752913
3,Saturday,9441216
4,Friday,9298152
5,Monday,9152400
6,Sunday,7537786


----
### 7.7 Regiões com as maiores gorgetas

In [40]:
con.execute("""
SELECT
l.Zone,
ROUND(AVG(f.tip_amount),2) gorjeta_media,
ROUND(AVG(f.total_amount),2) ticket_medio,
COUNT(*) corridas
FROM fato_corridas f
JOIN dim_localizacao l
ON f.PULocationID=l.LocationID
GROUP BY l.Zone
HAVING COUNT(*)>300
ORDER BY gorjeta_media DESC
LIMIT 10;
""").fetchdf()

,Zone,gorjeta_media,ticket_medio,corridas
0,JFK Airport,8.96,77.30,141746
1,LaGuardia Airport,8.71,65.26,88084
2,Outside of NYC,8.48,84.46,506
3,East Elmhurst,8.36,62.97,11968
4,Flushing Meadows-Corona Park,8.24,69.18,481
5,Baisley Park,7.75,73.02,837
6,Springfield Gardens South,6.80,71.85,387
7,South Ozone Park,6.64,64.34,549
8,Steinway,5.60,49.04,442
9,South Jamaica,5.56,62.08,512


----
### 7.8 zonas com mais corridas

In [46]:
con.execute("""
SELECT
    Borough,
    total_corridas,
    RANK() OVER(ORDER BY total_corridas DESC) AS ranking
FROM (
    SELECT
        l.Borough,
        COUNT(*) AS total_corridas
    FROM fato_corridas f
    JOIN dim_localizacao l
        ON f.PULocationID = l.LocationID
    GROUP BY l.Borough
)
ORDER BY ranking;
""").fetchdf()

,Borough,total_corridas,ranking
0,Manhattan,2496210,1
1,Queens,258610,2
2,Brooklyn,16968,3
3,Unknown,9510,4
4,Bronx,4970,5
5,N/A,506,6
6,EWR,53,7
7,Staten Island,31,8
